# TabICLv2 — Evaluation on Medical Datasets

## What is TabICLv2?

TabICLv2 is a pre-trained model specifically designed for tabular data (like CSV files with rows and columns). What makes it special is that it doesn't need to be trained on our data — it just reads all our training examples and uses them directly to make predictions.

In practice this means: no hyperparameter tuning, no long training, you just give it your data and it works. According to the original paper, it actually beats XGBoost and LightGBM on most benchmark datasets without any tuning at all, which is pretty impressive.

In this notebook, we run TabICLv2 on all our medical datasets automatically — you just need to set the path to your data folder and it takes care of the rest.


## 1. Install dependencies

We need to install the `tabicl` package. The first time you run this, it will also download the pre-trained model weights from HuggingFace (about 300 MB), so it might take a minute.


In [10]:
!pip install tabicl scikit-learn pandas numpy --quiet

## 2. Imports

Nothing special here — just the usual libraries plus `TabICLClassifier` from the tabicl package.


In [11]:
import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tabicl import TabICLClassifier

print("All imports OK ✓")

All imports OK ✓


## 3. Configuration — the only cell you need to edit

Change `DATA_DIR` to the path of your data folder, and `TARGET` to the name of your target column. Everything else can stay as is.

A quick note on `n_estimators`: TabICLv2 makes several predictions with slightly different random shuffles of the data and averages them. More estimators = slightly better results but slower. 8 is a good default.


In [23]:
# ============================================================
#  EDIT THESE VALUES TO MATCH YOUR SETUP
# ============================================================

DATA_DIR     = "/././DATA"   # path to your data folder
N_ESTIMATORS = 8                # number of ensemble predictions (8 is a good default)
DEVICE       = None             # None = auto-detect, "cuda" for GPU, "cpu" for CPU
TEST_SIZE    = 0.2              # train/test split ratio if no separate test file
RANDOM_STATE = 42

# default target column name — used for all datasets unless specified below
DEFAULT_TARGET = "Class"

# if some datasets have a different target column name, add them here
# format: "dataset_name": "column_name"
TARGET_PER_DATASET = {
    "diabetes": "hypo_next_day",
    "heart"   : "num",
}

# ============================================================

print(f"Data folder     : {Path(DATA_DIR).resolve()}")
print(f"Default target  : {DEFAULT_TARGET}")
print(f"Custom targets  : {TARGET_PER_DATASET}")
print(f"Estimators      : {N_ESTIMATORS}")
print(f"Device          : {DEVICE or 'auto'}")

Data folder     : /content/DATA
Default target  : Class
Custom targets  : {'diabetes': 'hypo_next_day', 'heart': 'num'}
Estimators      : 8
Device          : auto


## 4. Auto-detect datasets

This function scans your data folder and finds all datasets automatically. It looks for pairs of `*_train.csv` / `*_test.csv` files. If a dataset only has one file (no separate test split), it will create an 80/20 split automatically.

So if you add a new dataset to the folder later, you don't need to change anything in the code — it will just pick it up.


In [24]:
def discover_datasets(data_dir: str) -> dict:
    p = Path(data_dir)
    datasets = {}

    # Pattern 1: name_train.csv / name_test.csv  (e.g. heart_train.csv)
    train_suffix = {f.stem.replace("_train", ""): f for f in p.glob("*_train.csv")}
    test_suffix  = {f.stem.replace("_test",  ""): f for f in p.glob("*_test.csv")}
    for name in train_suffix:
        datasets[name] = {"train": train_suffix[name], "test": test_suffix.get(name)}

    # Pattern 2: train_name.csv / test_name.csv  (e.g. train_data.csv)
    already_found = set(train_suffix.values()) | set(test_suffix.values())
    train_prefix  = {f.stem.replace("train_", ""): f for f in p.glob("train_*.csv")}
    test_prefix   = {f.stem.replace("test_",  ""): f for f in p.glob("test_*.csv")}
    for name in train_prefix:
        if train_prefix[name] not in already_found:
            datasets[name] = {"train": train_prefix[name], "test": test_prefix.get(name)}

    # Pattern 3: train.csv / test.csv
    plain_train = p / "train.csv"
    plain_test  = p / "test.csv"
    if plain_train.exists() and plain_train not in already_found:
        datasets["dataset"] = {"train": plain_train, "test": plain_test if plain_test.exists() else None}

    # Pattern 4: single files with no split
    all_known = {info["train"] for info in datasets.values()} | {info["test"] for info in datasets.values() if info["test"]}
    for f in p.glob("*.csv"):
        if f not in all_known:
            datasets[f.stem] = {"train": f, "test": None}

    return datasets


datasets = discover_datasets(DATA_DIR)

print(f"Found {len(datasets)} dataset(s):\n")
for name, info in sorted(datasets.items()):
    test_label = info["test"].name if info["test"] else "auto split 80/20"
    print(f"  • {name:<20}  train: {info['train'].name:<30}  test: {test_label}")

Found 3 dataset(s):

  • diabetes              train: diabetes_train.csv              test: diabetes_test.csv
  • heart                 train: heart_train.csv                 test: heart_test.csv
  • hepatitis             train: hepatitis_train.csv             test: hepatitis_test.csv


## 5. Load and prepare the data

This function loads one dataset and prepares it for the model. The only thing we really need to do is encode the target column as numbers (e.g. the Hepatitis dataset uses 1/2 as labels instead of 0/1, so we convert that).

We don't need to normalize the features or fill in missing values ourselves — TabICLv2 handles all of that internally.


In [25]:
def load_dataset(info: dict, target: str) -> tuple:
    train_df = pd.read_csv(info["train"])

    if info["test"] is not None:
        test_df = pd.read_csv(info["test"])
    else:
        # stratified split to keep the same class proportions in train and test
        train_df, test_df = train_test_split(
            train_df,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=train_df[target],
        )

    if target not in train_df.columns:
        raise ValueError(
            f"Column '{target}' not found. Available columns: {list(train_df.columns)}"
        )

    feature_cols = [c for c in train_df.columns if c != target]

    X_train = train_df[feature_cols].values.astype(np.float32)
    X_test  = test_df[feature_cols].values.astype(np.float32)

    # encode labels as 0, 1, 2... (required by the model)
    le = LabelEncoder()
    y_train = le.fit_transform(train_df[target].values)
    y_test  = le.transform(test_df[target].values)

    return X_train, X_test, y_train, y_test, feature_cols, le


print("load_dataset function defined ✓")

load_dataset function defined ✓


## 6. Run TabICLv2

Here's where the magic happens. A quick note on what `.fit()` and `.predict()` actually do here, because it's a bit different from a usual model:

- **`.fit()`** is almost instant — it just stores the training data. No actual learning happens here.
- **`.predict()`** is where the model does its work. It reads all the training examples and uses them to predict the test labels in one go. This is the "in-context learning" part.

We compute several metrics: accuracy, balanced accuracy (useful when classes are imbalanced), F1 score, and ROC-AUC.

For binary datasets, we also sweep 100 threshold values between 0.1 and 0.9 directly on the test set and pick the one that maximizes F1-macro. The model is trained on the full training set — no validation split.

⚠️ **Important**: `f1_macro_best_thresh` is found on the test set itself, so it is optimistic by construction — treat it as an upper bound. Always report both values.


In [26]:
def find_best_threshold(y_true, y_proba):
    best_threshold = 0.5
    best_f1 = 0
    for threshold in np.linspace(0.1, 0.9, 100):
        y_pred = (y_proba[:, 1] >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    return round(float(best_threshold), 2), round(best_f1, 4)


def evaluate_tabicl(X_train, X_test, y_train, y_test) -> dict:

    clf = TabICLClassifier(
        n_estimators=N_ESTIMATORS,
        device=DEVICE,
        kv_cache=True,
        random_state=RANDOM_STATE,
    )

    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred_proba = clf.predict_proba(X_test)
    y_pred       = clf.predict(X_test)
    predict_time = time.perf_counter() - t1

    n_classes = y_pred_proba.shape[1]
    if n_classes == 2:
        auc = roc_auc_score(y_test, y_pred_proba[:, 1])
    else:
        auc = roc_auc_score(y_test, y_pred_proba, multi_class="ovr", average="macro")

    results = {
        "accuracy"          : round(accuracy_score(y_test, y_pred), 4),
        "balanced_accuracy" : round(balanced_accuracy_score(y_test, y_pred), 4),
        "f1_weighted"       : round(f1_score(y_test, y_pred, average="weighted", zero_division=0), 4),
        "f1_macro"          : round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
        "roc_auc"           : round(auc, 4),
        "fit_time_s"        : round(fit_time, 3),
        "predict_time_s"    : round(predict_time, 3),
        "total_time_s"      : round(fit_time + predict_time, 3),
    }

    if n_classes == 2:
        best_thresh, best_f1 = find_best_threshold(y_test, y_pred_proba)
        results["best_threshold"]      = best_thresh
        results["f1_macro_best_thresh"] = best_f1

    return results


print("evaluate_tabicl function defined ✓")

evaluate_tabicl function defined ✓


## 7. Run on all datasets

We loop over all detected datasets, load them, and evaluate TabICLv2 on each one. Results are stored as we go.


In [27]:
all_results = []

# metrics printed during the loop (threshold=0.5 only, clean comparison with other models)
DISPLAY_METRICS = ["accuracy", "balanced_accuracy", "f1_weighted", "f1_macro", "roc_auc",
                   "fit_time_s", "predict_time_s", "total_time_s"]

for name, info in sorted(datasets.items()):
    print(f"\n{'─'*55}")
    print(f"  Dataset : {name.upper()}")
    print(f"  Train   : {info['train'].name}")
    print(f"  Test    : {info['test'].name if info['test'] else 'auto split 80/20'}")
    print(f"{'─'*55}")

    try:
        target = TARGET_PER_DATASET.get(name, DEFAULT_TARGET)
        print(f"  Target column : {target}")

        X_train, X_test, y_train, y_test, features, le = load_dataset(info, target)

        print(f"  Train shape : {X_train.shape}  |  Test shape : {X_test.shape}")
        print(f"  Classes     : {list(le.classes_)} → {list(range(len(le.classes_)))}")
        print(f"  Class distribution (train) : {dict(zip(le.classes_, np.bincount(y_train)))}")
        print(f"  Running TabICLv2...")

        metrics = evaluate_tabicl(X_train, X_test, y_train, y_test)

        print(f"\n  {'Metric':<28} {'Value':>8}")
        print(f"  {'─'*38}")
        for k, v in metrics.items():
            if k in DISPLAY_METRICS:
                print(f"  {k:<28} {v:>8}")

        all_results.append({"dataset": name, "target": target, **metrics})

    except Exception as e:
        print(f"  ✗ Error on {name}: {e}")
        try:
            cols = list(pd.read_csv(info['train'], nrows=0).columns)
            print(f"     Available columns: {cols}")
            print(f"     → Add  \"{name}\": \"your_target_col\"  to TARGET_PER_DATASET in cell 3")
        except:
            pass

print(f"\nDone — {len(all_results)}/{len(datasets)} dataset(s) completed successfully.")


───────────────────────────────────────────────────────
  Dataset : DIABETES
  Train   : diabetes_train.csv
  Test    : diabetes_test.csv
───────────────────────────────────────────────────────
  Target column : hypo_next_day
  Train shape : (3104, 14)  |  Test shape : (776, 14)
  Classes     : [np.int64(0), np.int64(1)] → [0, 1]
  Class distribution (train) : {np.int64(0): np.int64(2873), np.int64(1): np.int64(231)}
  Running TabICLv2...
INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.



tabicl-classifier-v2-20260212.ckpt:   0%|          | 0.00/110M [00:00<?, ?B/s]


  Metric                          Value
  ──────────────────────────────────────
  accuracy                       0.9317
  balanced_accuracy              0.5431
  f1_weighted                    0.9042
  f1_macro                       0.5616
  roc_auc                        0.8829
  fit_time_s                    127.832
  predict_time_s                  44.06
  total_time_s                  171.892

───────────────────────────────────────────────────────
  Dataset : HEART
  Train   : heart_train.csv
  Test    : heart_test.csv
───────────────────────────────────────────────────────
  Target column : num
  Train shape : (228, 13)  |  Test shape : (58, 13)
  Classes     : [np.int64(0), np.int64(1)] → [0, 1]
  Class distribution (train) : {np.int64(0): np.int64(125), np.int64(1): np.int64(103)}
  Running TabICLv2...

  Metric                          Value
  ──────────────────────────────────────
  accuracy                       0.8448
  balanced_accuracy              0.8413
  f1_weighted 

## 8. Summary table

Let's put all the results in a clean table to compare across datasets.


In [28]:
if all_results:
    df_results = pd.DataFrame(all_results).set_index("dataset")

    metric_cols = ["accuracy", "balanced_accuracy", "f1_weighted", "f1_macro", "roc_auc"]
    time_cols   = ["fit_time_s", "predict_time_s", "total_time_s"]
    thresh_cols = [c for c in ["best_threshold", "f1_macro_best_thresh"] if c in df_results.columns]

    print("=" * 55)
    print("  PERFORMANCE METRICS — TabICLv2 (threshold = 0.5)")
    print("=" * 55)
    print(df_results[metric_cols].to_string())

    if thresh_cols:
        print("\n" + "=" * 55)
        print("  BEST THRESHOLD (swept on test set, 100 values)")
        print("  f1_macro_best_thresh = best F1-macro achievable on test")
        print("=" * 55)
        print(df_results[thresh_cols].to_string())

    print("\n" + "=" * 55)
    print("  EXECUTION TIME (seconds)")
    print("=" * 55)
    print(df_results[time_cols].to_string())
    print()
    print("Note: fit_time is near 0 because fit() only stores the data.")
    print("      The actual computation happens in predict_time.")

  PERFORMANCE METRICS — TabICLv2 (threshold = 0.5)
           accuracy  balanced_accuracy  f1_weighted  f1_macro  roc_auc
dataset                                                               
diabetes     0.9317             0.5431       0.9042    0.5616   0.8829
heart        0.8448             0.8413       0.8445    0.8425   0.9123
hepatitis    0.6774             0.5387       0.6774    0.5387   0.7679

  BEST THRESHOLD (swept on test set, 100 values)
  f1_macro_best_thresh = best F1-macro achievable on test
           best_threshold  f1_macro_best_thresh
dataset                                        
diabetes             0.18                0.7062
heart                0.53                0.8594
hepatitis            0.79                0.7163

  EXECUTION TIME (seconds)
           fit_time_s  predict_time_s  total_time_s
dataset                                            
diabetes      127.832          44.060       171.892
heart           5.049           1.891         6.940
hepatitis 

## 9. Save results

We save everything to a CSV file so we can compare with the other models (XGBoost, LightGBM, TabPFN, TabLLM...) later in the global benchmark comparison.


In [29]:
if all_results:
    output_path = Path(DATA_DIR).parent / "results_tabicl.csv"
    df_results.to_csv(output_path)
    print(f"✓ Results saved to: {output_path}")

✓ Results saved to: /content/results_tabicl.csv
